<a href="https://colab.research.google.com/github/SWATHIRAVIRS/PDF-analyser/blob/main/LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install langchain faiss-cpu sentence-transformers pypdf langchain_community

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/EXP 8 - BUILDING AND VALIDATING LOGISTIC MODELS (1).pdf")
documents = loader.load()

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = splitter.split_documents(documents)

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(docs, embeddings)

In [ ]:
query = "What is this document about?"
results = db.similarity_search(query,k=3)
context="\n".join([doc.page_content for doc in results])
print(results[0].page_content)

In [ ]:
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

pipe = pipeline("text-generation", model="gpt2")
llm = HuggingFacePipeline(pipeline=pipe)

answer = llm.invoke(results[0].page_content)
print(answer)

In [ ]:
prompt = f"""
Answer the question based only on the context below:

Context:
{context}

Question:
{query}
"""

In [ ]:
prompt = f"""
You are a helpful assistant.

Answer the question clearly and concisely using only the context below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question:
{query}
"""

In [ ]:
chat_history = ""

while True:
    query = input("You: ")

    results = db.similarity_search(query, k=3)
    context = "\n".join([doc.page_content for doc in results])

    prompt = f"""
    Chat History:
    {chat_history}

    Context:
    {context}

    Question:
    {query}
    """

    answer = llm.invoke(prompt)
    print("Bot:", answer)

    chat_history += f"\nUser: {query}\nBot: {answer}"